In [1]:
import pandas as pd
import numpy as np

X_train = pd.read_csv("../data/preprocessed/X_train_pca.csv")
X_test = pd.read_csv("../data/preprocessed/X_test_pca.csv")

y_train = pd.read_csv("../data/preprocessed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/preprocessed/y_test.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2496, 80)
X_test shape: (278, 80)
y_train shape: (2496,)
y_test shape: (278,)


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

dnn_model = Sequential([
    Input(shape=(80,)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1)
])

dnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        10,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,737 (81.00 KB)

 Trainable params: 20,737 (81.00 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
dnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

print("DNN compiled successfully!")

DNN compiled successfully!


In [4]:
history = dnn_model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

print("DNN training completed!")

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 5.9416 - mae: 1.7543 - val_loss: 2.0300 - val_mae: 0.9447
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.0801 - mae: 0.8694 - val_loss: 1.6075 - val_mae: 0.8017
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.7628 - mae: 0.7613 - val_loss: 1.4784 - val_mae: 0.7641
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.6572 - mae: 0.7150 - val_loss: 1.3962 - val_mae: 0.7359
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.5805 - mae: 0.6807 - val_loss: 1.3372 - val_mae: 0.7146
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.5180 - mae: 0.6541 - val_loss: 1.2997 - val_mae: 0.7046
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.4685 - mae: 0.6348 - val_loss: 1.2726 - val_mae: 0.6976
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.4251 - mae: 0.6178 - val_loss: 1.2627 - val_mae: 0.6962
Epoch 9/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.38

In [5]:
y_pred_dnn = dnn_model.predict(X_test).flatten()

print("First 10 DNN predictions:")
print(y_pred_dnn[:10])

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
First 10 DNN predictions:
[3.6162286 3.4519336 4.696898  3.6938753 4.90908   3.9692616 5.1947002
 3.8021336 3.5169    4.553208 ]


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_dnn = mean_absolute_error(y_test, y_pred_dnn)
rmse_dnn = np.sqrt(mean_squared_error(y_test, y_pred_dnn))
r2_dnn = r2_score(y_test, y_pred_dnn)

print("DNN Performance:")
print("MAE :", mae_dnn)
print("RMSE:", rmse_dnn)
print("R²  :", r2_dnn)

DNN Performance:
MAE : 0.8492945315422074
RMSE: 1.3836948251195116
R²  : 0.1610529451744388


In [7]:
dnn_results = pd.DataFrame({
    "Model": ["DNN"],
    "MAE": [mae_dnn],
    "RMSE": [rmse_dnn],
    "R2": [r2_dnn]
})

dnn_results.to_csv("../results/dnn_results.csv", index=False)

print(dnn_results)
print("\nDNN results saved successfully!")

  Model       MAE      RMSE        R2
0   DNN  0.849295  1.383695  0.161053

DNN results saved successfully!


In [8]:
dnn_model.save("../models/dnn_model.keras")

print("DNN model saved successfully!")

DNN model saved successfully!
